<a href="https://colab.research.google.com/github/nav-in27/CUSTOMER-CHURN-PREDICTION_USING-AUTOGLUON/blob/main/Customer_churn_prediction_using_autogluon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MODULE 1: Installation & Environment Setup

In [ ]:
!pip install -q pandas==2.2.2
!pip install -q "autogluon.tabular[all]" scikit-learn joblib shap

In [ ]:
!pip -q install autogluon.tabular==1.1.0 scikit-learn pandas numpy joblib shap

ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.6, <3.9; 0.1.0b20210207 Requires-Python >=3.6, <3.8; 0.1.0b20210208 Requires-Python >=3.6, <3.8; 0.1.0b20210209 Requires-Python >=3.6, <3.8; 0.1.0b20210210 Requires-Python >=3.6, <3.8; 0.1.0b20210211 Requires-Python >=3.6, <3.8; 0.1.0b20210212 Requires-Python >=3.6, <3.8; 0.1.0b20210213 Requires-Python >=3.6, <3.8; 0.1.0b20210214 Requires-Python >=3.6, <3.8; 0.1.0b20210215 Requires-Python >=3.6, <3.8; 0.1.0b20210216 Requires-Python >=3.6, <3.8; 0.1.0b20210217 Requires-Python >=3.6, <3.8; 0.1.0b20210218 Requires-Python >=3.6, <3.8; 0.1.0b20210219 Requires-Python >=3.6, <3.8; 0.1.0b20210220 Requires-Python >=3.6, <3.8; 0.1.0b20210221 Requires-Python >=3.6, <3.8; 0.1.0b20210222 Requires-Python >=3.6, <3.8; 0.1.0b20210223 Requires-Python >=3.6, <3.8; 0.1.0b20210224 Requires-Python >=3.6, <3.8; 0.1.0b20210225 Requires-Python >=3.6, <3.9; 0.1.0b20210226 Requires-Python >=3.6, <3.9; 0.1.0b

In [ ]:
import pandas as pd, numpy as np, warnings, joblib, time, json
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, f1_score
from autogluon.tabular import TabularPredictor
warnings.filterwarnings("ignore")

MODULE 2: Data Loading & Preprocessing




In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

Saving telecom_customer_churn.csv to telecom_customer_churn (1).csv


In [ ]:
df.columns = df.columns.str.strip().str.lower()


In [ ]:
target_col = "customer status"

df[target_col] = df[target_col].str.strip().str.lower()
df[target_col] = df[target_col].map({
    'churned': 1,
    'stayed': 0,
    'joined': 0   # Or remove these rows later
})


In [ ]:
leak_cols = [
    'churn reason',
    'churn category',
    'customer id'
]

df = df.drop(columns=[c for c in leak_cols if c in df.columns])


In [ ]:
for col in df.columns:
    if col != target_col and ('charge' in col or 'tenure' in col or
                              'month' in col or 'age' in col):
        df[col] = pd.to_numeric(df[col], errors='coerce')


In [ ]:
initial_len = len(df)

df = df.dropna(subset=[target_col])   # remove missing labels
df = df.dropna(axis=0, how='all')     # remove empty rows

print(f" Cleaned data: {len(df)} rows ({initial_len - len(df)} removed)")


 Cleaned data: 7043 rows (0 removed)


In [ ]:
class_counts = df[target_col].value_counts().sort_index()
print("Class distribution:\n", class_counts)


Class distribution:
 customer status
0    5174
1    1869
Name: count, dtype: int64


In [ ]:
n_classes = len(class_counts)
problem_type = 'binary' if n_classes == 2 else 'multiclass'
print(f"Detected {problem_type} problem ({n_classes} classes)")


Detected binary problem (2 classes)


In [ ]:
for col in df.columns:
    if col != target_col and ('charge' in col or 'tenure' in col or 'month' in col or 'age' in col):
        df[col] = pd.to_numeric(df[col], errors='coerce')

MODULE 3: Problem Understanding & Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[target_col]
)

print(f"✓ Train: {len(train)} | Test: {len(test)}")


✓ Train: 5634 | Test: 1409


In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df[target_col])
print(f"✓ Train: {len(train)} | Test: {len(test)}")

✓ Train: 5634 | Test: 1409


MODULE 4: Model Building Using AutoGluon

In [ ]:
from autogluon.tabular import TabularPredictor
import time

print(f"\nTraining AutoGluon for {problem_type} classification...")
start = time.time()

predictor = TabularPredictor(
    label=target_col,
    eval_metric='roc_auc' if problem_type=='binary' else 'accuracy',
    problem_type=problem_type,
    path='best_churn_model'
).fit(
    train,
    time_limit=600,
    presets='best_quality',
    verbosity=2
)

training_time = (time.time() - start) / 60
print(f"✓ Training finished in {training_time:.1f} min")


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       9.32 GB / 12.67 GB (73.5%)
Disk Space Avail:   67.64 GB / 112.64 GB (60.0%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdout


Training AutoGluon for binary classification...


Leaderboard on holdout data (DyStack):
                     model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0        LightGBMXT_BAG_L2       0.909979   0.908481     roc_auc        2.792211       1.221082  126.878988                 0.107265                0.118123          40.100955            2       True          5
1          LightGBM_BAG_L1       0.909193   0.912057     roc_auc        0.159145       0.196283   41.779433                 0.159145                0.196283          41.779433            1       True          2
2      WeightedEnsemble_L2       0.909128   0.913288     roc_auc        2.688173       1.104367   86.846234                 0.003227                0.001408           0.068201            2       True          4
3      WeightedEnsemble_L3       0.909128   0.913288     roc_auc        2.689911       1.104334   86.890471          

✓ Training finished in 10.5 min


MODULE 5: Model Evaluation

In [ ]:
print("\n Evaluating on test set...")
test_data = test.drop(columns=[target_col])
preds = predictor.predict(test_data)


if problem_type == 'binary':
    proba_df = predictor.predict_proba(test_data)
    if isinstance(proba_df, pd.DataFrame):
        proba = proba_df[1].values if 1 in proba_df.columns else proba_df.iloc[:, 1].values
    else:
        proba = proba_df
    roc = roc_auc_score(test[target_col], proba)
    f1 = f1_score(test[target_col], preds)
    print(f"\n{'='*50}")
    print(f"Final Scores → ROC-AUC: {roc:.3f}  |  F1: {f1:.3f}")
    print(f"{'='*50}\n")
else:

    from sklearn.metrics import accuracy_score
    acc = accuracy_score(test[target_col], preds)
    print(f"\n{'='*50}")
    print(f"Final Accuracy: {acc:.3f}")
    print(f"{'='*50}\n")

print(classification_report(test[target_col], preds, digits=3))


 Evaluating on test set...

Final Scores → ROC-AUC: 0.920  |  F1: 0.710

              precision    recall  f1-score   support

           0      0.888     0.916     0.902      1035
           1      0.745     0.679     0.710       374

    accuracy                          0.853      1409
   macro avg      0.816     0.798     0.806      1409
weighted avg      0.850     0.853     0.851      1409



In [ ]:
print("\n Top 3 Churn Drivers:")
try:
    importance = predictor.feature_importance(test)
    top3 = importance.head(3)
    for i, (feat, score) in enumerate(top3.iterrows(), 1):
        print(f"  {i}. {feat:30s} (importance: {score['importance']:.4f})")
except Exception as e:
    print(f"  Unable to compute feature importance: {e}")

print("\n Pipeline complete!")

Computing feature importance via permutation shuffling for 34 features using 1409 rows with 5 shuffle sets...



 Top 3 Churn Drivers:


	226.53s	= Expected runtime (45.31s per shuffle set)
	102.97s	= Actual runtime (Completed 5 of 5 shuffle sets)


  1. contract                       (importance: 0.0948)
  2. number of referrals            (importance: 0.0382)
  3. number of dependents           (importance: 0.0157)

 Pipeline complete!


In [ ]:
!pip install -q "autogluon.tabular[all]" scikit-learn joblib shap streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 117.3 MB/s eta 0:00:00


In [ ]:
%%writefile churn_app.py
import os

import pandas as pd
import pandas.api.types as pdtypes
import streamlit as st
from autogluon.tabular import TabularPredictor

# ----------------- PAGE CONFIG -----------------
st.set_page_config(
    page_title="Telecom Customer Churn Predictor",
    layout="wide",
)

st.title("📞 Telecom Customer Churn Predictor")
st.write(
    "Predict whether a telecom customer is likely to churn based on a few key attributes. "
    "The model under the hood is an AutoGluon Tabular model you trained earlier."
)

# ----------------- MODEL & DATA LOADING -----------------
@st.cache_resource
def load_model_and_training_info():
    """
    Loads the AutoGluon model saved in 'best_churn_model' and
    uses its training data to derive:
      - feature list (excluding leakage columns and target)
      - default values for each feature
      - target column name and problem type
    """
    model_path = "best_churn_model"
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"AutoGluon model directory '{model_path}' not found. "
            f"Train and save the model first in this working directory."
        )

    predictor = TabularPredictor.load(model_path)

    # Load training data used by AutoGluon
    train_loaded = predictor.load_data_internal("train")
    # Sometimes AutoGluon returns (X, y); handle both cases
    if isinstance(train_loaded, tuple):
        train = train_loaded[0]
    else:
        train = train_loaded

    target_col = predictor.label
    problem_type = predictor.problem_type

    # Columns we don't want to ask the user for (leakage / ID / label)
    leakage_cols = ["customer id", "churn reason", "churn category"]
    leakage_cols = [c for c in leakage_cols if c in train.columns]

    feature_cols = [
        c for c in train.columns
        if c not in leakage_cols + [target_col]
    ]

    # Default values taken from the training data
    default_values = {}
    for col in feature_cols:
        if pdtypes.is_numeric_dtype(train[col]):
            default_values[col] = float(train[col].mean())
        else:
            # mode()[0] can fail if column is all NaN, so be safe
            mode = train[col].mode(dropna=True)
            default_values[col] = mode.iloc[0] if not mode.empty else ""

    return predictor, train, feature_cols, default_values, target_col, problem_type


try:
    predictor, train_df, feature_cols, default_values, target_col, problem_type = load_model_and_training_info()
    st.success(" AutoGluon model loaded successfully.")
except Exception as e:
    st.error(f" Failed to load model: {e}")
    st.stop()

# ----------------- WHICH FEATURES TO ASK USER FOR -----------------
# You can change this list to show more / fewer fields in the UI.
user_requested_features = [
    "tenure in months",
    "number of dependents",
    "number of referrals",
    "monthly charge",
    "total extra data charges",
]

available_features = [f for f in user_requested_features if f in feature_cols]

if not available_features:
    st.warning(
        "None of the requested features exist in the training data. "
        "Please check feature names in `user_requested_features`."
    )
    st.stop()

st.header(" Customer Information")

# A few sensible ranges for numeric inputs (tweak as needed)
feature_specs = {
    "tenure in months": {"min": 0, "max": 72, "step": 1.0},
    "number of dependents": {"min": 0, "max": 10, "step": 1.0},
    "number of referrals": {"min": 0, "max": 10, "step": 1.0},
    "monthly charge": {"min": 0.0, "max": 120.0, "step": 0.1},
    "total extra data charges": {"min": 0.0, "max": 200.0, "step": 0.1},
}

user_values = {}

with st.form("churn_form"):
    cols = st.columns(len(available_features))

    for i, feature in enumerate(available_features):
        default_val = default_values.get(feature, 0.0)
        col = cols[i]

        label = feature.title()

        # Assume these requested features are numeric; for other
        # column types you'd use selectbox, checkbox, etc.
        spec = feature_specs.get(feature, None)
        if spec is not None:
            user_val = col.number_input(
                label,
                min_value=float(spec["min"]),
                max_value=float(spec["max"]),
                step=float(spec["step"]),
                value=float(default_val),
            )
        else:
            user_val = col.number_input(label, value=float(default_val))

        user_values[feature] = user_val

    submitted = st.form_submit_button(" Predict Churn")

# ----------------- PREDICTION -----------------
if submitted:
    # Start with all default feature values
    input_dict = default_values.copy()

    # Override only the features the user actually set in the form
    for feature, val in user_values.items():
        input_dict[feature] = val

    # Build a single-row DataFrame in the correct column order
    input_df = pd.DataFrame([input_dict], columns=feature_cols)

    # Make sure numeric columns are numeric
    for col in input_df.columns:
        if pdtypes.is_numeric_dtype(train_df[col]):
            input_df[col] = pd.to_numeric(input_df[col], errors="coerce")
            if input_df[col].isnull().any():
                # Fallback to training mean if conversion failed
                input_df[col] = default_values[col]

    # Prediction
    pred = predictor.predict(input_df)[0]
    proba_df = predictor.predict_proba(input_df)

    st.subheader(" Prediction Result")

    if problem_type == "binary":
        # AutoGluon will usually use the label values as column names (0 and 1)
        churn_prob = None
        if 1 in proba_df.columns:
            churn_prob = proba_df[1].iloc[0]
        elif "churned" in proba_df.columns:
            churn_prob = proba_df["churned"].iloc[0]
        else:
            # fallback: last column
            churn_prob = proba_df.iloc[0, -1]

        retain_prob = 1.0 - churn_prob

        if pred == 1 or str(pred).lower() in ["churned", "yes"]:
            st.error(f" Predicted Churn: **YES**\n\nEstimated churn probability: **{churn_prob:.2%}**")
        else:
            st.success(f" Predicted Churn: **NO**\n\nEstimated churn probability: **{churn_prob:.2%}**")

        st.progress(float(churn_prob))
        st.caption(f"Churn: {churn_prob:.2%} • Stay: {retain_prob:.2%}")

    else:
        # Multiclass fallback
        pred_prob = proba_df[pred].iloc[0] if pred in proba_df.columns else proba_df.iloc[0].max()
        st.info(f"Predicted class: **{pred}** (probability: {pred_prob:.2%})")
        st.dataframe(proba_df)

st.markdown(
    """
THIS PROJECT HAS BEEN DEVELOPED BY
NAVEEN , SHARUKK , MANTRAA AND GNANAMAHADEVI.
"""
)


Writing churn_app.py


In [ ]:
import os

!pip install -q pyngrok

from pyngrok import ngrok
import threading, subprocess, time

# 1. Kill any previous tunnels that may be hanging
ngrok.kill()

# Set your ngrok authtoken here. Replace 'YOUR_AUTHTOKEN' with your actual token.
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("34iMgGcRGt9zbWAB1O0UaRg2wY2_2CQEmWEGrSDunrVJ4n2eh")

PORT = 8501

def run_streamlit():
    # Run the app file we created above
    subprocess.run(
        ["streamlit", "run", "churn_app.py",
         "--server.port", str(PORT),
         "--server.address", "0.0.0.0"],
        check=True
    )

# 2. Start Streamlit in a background thread
thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# 3. Wait a bit for Streamlit to boot
time.sleep(5)

# 4. Create the tunnel and show public URL
public_url = ngrok.connect(PORT).public_url
print(" Public URL:", public_url)


 Public URL: https://unhelpful-wavelessly-calista.ngrok-free.dev
